In [11]:
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
from scipy import sparse
from underthesea import sent_tokenize
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
import re
import string

In [6]:
# load dữ liệu train 
path = "/data/dataset_news_summary/text_rank_summary/json/"
file = 'dataset_train.json'
df = pd.read_json(path + file)

In [7]:
print(df.head())

                                               Title  \
0  Lần thứ 3 gia hạn nộp thuế: Thêm động lực để D...   
1  Doanh nghiệp sẵn sàng chi tiền tiêm vaccine ng...   
2  Tiền Giang: Gần 740 trường hợp về từ Đà Nẵng đ...   
3  Vay 5 triệu, thiếu nữ bị chủ nợ đánh đập và gi...   
4  Mặt trái những ứng dụng hẹn hò: Booking girl –...   

                                             Summary  \
0  Quyết định gia hạn thuế và tiền thuê đất trong...   
1  Một số doanh nghiệp trong khu công nghiệp sẵn ...   
2  Thông tin từ Sở Y tế tỉnh Tiền Giang ngày 10.8...   
3  Sau khi vay 5 triệu đồng, thiếu nữ bị nhóm tha...   
4  Thời bùng nổ mạng xã hội, ngoài những phần mềm...   

                                            Contents    Category  
0  Tăng đề kháng cho doanh nghiệp Mặc dù đã đạt đ...  Kinh doanh  
1  Công nhân lo lắng Đề xuất của Tổng Liên đoàn L...   Công đoàn  
2  Số trường hợp còn lại đang được cách ly theo d...      Xã hội  
3  Công an huyện Hưng Nguyên, Nghệ An vừa bắt giữ...   Phá

In [8]:
# xây dựng kho TF-IDF
vectorizer = TfidfVectorizer(
    max_features=50000,       # lấy khoảng 50k từ thường xuyên
    ngram_range=(1,2),
    min_df=3,                 # bỏ từ xuất hiện quá ít
    max_df=0.9,               # bỏ từ quá phổ biến
    lowercase=True
)
X_train = vectorizer.fit_transform(df['Contents'].astype(str))

In [9]:
print("Train shape:", X_train.shape)

Train shape: (220520, 50000)


In [14]:
# lưu matrix train TF_IDF 
sparse.save_npz("/data/dataset_news_summary/tf_idf/matrix/tfidf_train_matrix.npz", X_train) # lưu ma trix

# lưu vectorizer biểu diễn
joblib.dump(vectorizer, "/data/dataset_news_summary/tf_idf/vector/tfidf_vectorizer.pkl")

['/data/dataset_news_summary/tf_idf/vector/tfidf_vectorizer.pkl']

In [2]:
# load vectorizer đã xây dựng và lưu lại từ tập train
vectorizer = joblib.load("/data/dataset_news_summary/tf_idf/vector/tfidf_vectorizer.pkl")

In [3]:
# Load dataset test
path = "/data/dataset_news_summary/text_rank_summary/json/"
file = 'dataset_test.json'
df = pd.read_json(path + file)

In [4]:
print(df.head())

                                               Title  \
0  Giá vàng hôm nay 11.11: Trượt khỏi đà tăng, li...   
1  Thủ tướng chủ trì họp Thường trực Chính phủ về...   
2  Cùng một ngày, bác sĩ Hoàng Công Lương nhận 2 ...   
3  Khẳng định vị thế, uy tín và năng lực của Việt...   
4  Vietnam Airlines hoàn tất việc chuẩn bị bay th...   

                                             Summary  \
0  Giá vàng hôm nay 11.11: Vàng trên thị trường q...   
1  Chiều 24.6, Thủ tướng Nguyễn Xuân Phúc đã chủ ...   
2  Trong một ngày, bác sĩ Hoàng Công Lương bất ng...   
3  Việt Nam chính thức đảm nhận vai trò Ủy viên k...   
4  Ngày 21.9.2021, Vietnam Airlines chính thức ho...   

                                            Contents    Category  
0  Giá vàng trong nước Công ty VBĐQ Sài Gòn (SJC)...  Kinh doanh  
1  Phát biểu mở đầu cuộc họp, Thủ tướng cho biết,...     Thời sự  
2  Mới đây, mạng xã hội lan truyền hình ảnh hai l...   Pháp luật  
3  Năm hết sức bận rộn Công việc của Việt Nam khi...    Th

In [9]:
# hàm tách câu
# tách câu trong văn bản thành 1 list, loại bỏ những caai ít hơn 10 từ vì ít thông tin
def sentence_tokenize(text):
    sentences = sent_tokenize(text)
    return [s.strip() for s in sentences if len(s.strip()) > 10]

In [12]:
# hàm clean_text cho từng câu 
def clean_text(text):
    # Lowercase
    text = text.lower()

    # Loại bỏ URL
    text = re.sub(r'http\S+|www\S+', '', text)

    # Loại bỏ số
    text = re.sub(r'\d+', ' ', text)

    # Loại bỏ ký tự đặc biệt (giữ lại chữ cái tiếng Việt)
    text = re.sub(rf"[{re.escape(string.punctuation)}]", " ", text)

    # Loại bỏ ký tự thừa
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [58]:
# hàm tính page rank và lựa chọn k-top
def text_rank_summary(text, top_n=5):
    # tách các câu
    sentences = sentence_tokenize(text)

    if len(sentences) <= top_n:
        return " ".join(sentences)

    # Clean từng câu
    clean_sentences = [clean_text(s) for s in sentences]

    # Vectorize
    tfidf_sent = vectorizer.transform(clean_sentences)

    # Similarity tính độ tương đồng giữa từng cặp câu
    similarity_matrix = cosine_similarity(tfidf_sent)
    np.fill_diagonal(similarity_matrix, 0)

    # normalize chuẩn hóa lại matrix điểm số
    row_sum = similarity_matrix.sum(axis=1, keepdims=True)
    similarity_matrix = similarity_matrix / (row_sum + 1e-8)

    # Graph(tạo đồ thị giữa các câu)
    graph = nx.from_numpy_array(similarity_matrix)

    # PageRank (dùng thư viện)
    scores = nx.pagerank(graph)

    # Ranking
    ranked = sorted(
        ((scores[i], i, s) for i, s in enumerate(sentences)),
        reverse=True
    )

    # Giữ thứ tự gốc
    top_sentences = sorted(ranked[:top_n], key=lambda x: x[1])

    summary = " ".join([s for (_, _, s) in top_sentences])

    return summary

In [59]:
df_new_test = df.sample(n=1000, random_state=42)

In [63]:
# chạy multiprocessing giúp nhanh hơn
def process_textrank(text):
    try:
        return text_rank_summary(text, top_n=3)
    except:
        return ""

In [61]:
# dùng pool mở nhiều worker
import multiprocessing as mp
from tqdm import tqdm

def parallel_textrank(texts, n_workers=4):

    with mp.Pool(processes=n_workers) as pool:
        results = list(
            tqdm(pool.imap(process_textrank, texts),
                 total=len(texts))
        )

    return results

In [64]:
if __name__ == "__main__":
    texts = df_new_test["Contents"].tolist()
    
    # chạy trên 8 worker
    summaries = parallel_textrank(texts, n_workers=8)
    
    df_new_test["textrank_summary"] = summaries

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 2935.08it/s]


In [65]:
df_new_test

,Title,Summary,Contents,Category,textrank_summary
22396,Vay tiền qua app: Bí ẩn văn phòng làm việc của...,Nhiều app cho vay có văn phòng làm việc rất bí...,"Doạ chặt ngón tay ""con nợ"" Phản ánh đến Báo La...",Xã hội,"Nhanh lên chuyển qua liền ngay đây, 10 phút sa..."
17644,Nông dân làm gì để vay được vốn làm nông nghiệ...,"Ngày 13.10, tại Diễn đàn Nông dân quốc gia lần...","""Sản xuất nông nghiệp công nghệ cao phải khẳng...",Kinh doanh,Ví dụ như dự án có các hợp đồng tiêu thụ ổn đị...
16852,Chế độ ăn ảnh hưởng trực tiếp tới tương lai hà...,Theo một nghiên cứu mang tính bước ngoặt được ...,Một nhóm gồm 30 nhà nghiên cứu kết luận trên t...,Thế giới,Một nhóm gồm 30 nhà nghiên cứu kết luận trên t...
21304,"Hà Nội ghi nhận 1.866 ca COVID-19 mới, gần 700...","Hà Nội - Theo Sở Y tế Hà Nội, từ 18h ngày 29.1...",Phân bố theo nơi ghi nhận như sau: Tại cộng đồ...,Xã hội,Trong đó 699 ca cộng đồng ghi nhận tại 233 xã ...
7482,"Tin tức pháp luật 24h: Lừa đảo “chạy” việc, ng...",Nguyên hiệu trưởng nhận tổng số tiền 890 triệu...,Khởi tố hiệu trưởng nhận hơn 1 tỉ đồng lừa xin...,Pháp luật,Khởi tố người đâm chết bạn nhậu vì vỗ mông vợ ...
...,...,...,...,...,...
10137,Quảng Bình: Cách ly người từ các tỉnh có bệnh ...,"Ngày 11.4 UBND tỉnh Quảng Bình cho biết, vừa c...",Công văn do Phó Chủ tịch UBND tỉnh Quảng Bình ...,Xã hội,Công văn do Phó Chủ tịch UBND tỉnh Quảng Bình ...
21396,Liên tiếp sạt lở khốc liệt ở miền Trung: Cảnh ...,"Mưa lũ dữ dội ở miền Trung, sạt lở đất đã và đ...","""Đưa ra bản đồ cảnh báo sạt lở, nhưng tỉnh Thừ...",Xã hội,"""Đưa ra bản đồ cảnh báo sạt lở, nhưng tỉnh Thừ..."
19971,Hải Phòng: Nữ sinh bị đuổi khỏi KTX vì nói chu...,Nữ sinh viên Trường ĐH Y dược Hải Phòng bị đuổ...,"Ngày 24.12, phản ánh với Lao Động, L.T.L. (21 ...",Bạn đọc,"(21 tuổi, sinh viên năm 3 Trường đại học Y dượ..."
16454,"Tìm hướng mở rộng, tăng lượng việc làm từ kinh...","Dù đang thu hút, tạo việc làm cho 6,8 triệu th...",Sẽ tăng số thành viên thêm 10% Trong 5 tháng đ...,Kinh doanh,Sẽ tăng số thành viên thêm 10% Trong 5 tháng đ...


In [66]:
df_new_test.to_json("/data/dataset_news_summary/tf_idf/test/test_tf_idf.json",
              orient="records", force_ascii=False,indent=2)